In [1]:
import yaml
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Leer credenciales del arhivo YAML
with open("credenciales.yaml", "r") as file:
     config = yaml.safe_load(file)

db = config["mysql"]

# Crear conexión
# Su función es crear un Engine, que es el objeto que SQLAlchemy utiliza para comunicarse con la base de datos.
engine = create_engine(                                     
    f"mysql+pymysql://{db['user']}:{db['password']}@"
    f"{db['host']}:{db['port']}/{db['database']}"
)

##### ANALISIS DE TEXTO

##### 5-1 Introduccion

Se utilizaran datos de reseñas de un parque de diversiones. El objetivo es aprender herramientas para limpieza y análisis básico de texto. 


##### 5-2 Qué es el analisis de texto

Los datos de tipo texto pueden ser: 

$\cdot$ Estructurados: 		Datos de formularios por ejemplo

$\cdot$ No estructurados:	Datos extraídos directamente de libros, revistas, páginas web.

$\cdot$ Semi estructurados:	Tienen parte estructurada y no estructurada como los correos electronicos.


Hay 2 tipos de análisis de texto:

$\cdot$ Cualitativo:	Leer el texto y hacer un resumen del contenido.

$\cdot$ Cuantitativo:	Categorización, extracción de información. SQL nos servira para este análisis.

Metas:

$\cdot$ Extracción de texto: Extraer el numero de estrellas que dejo una reseña.

$\cdot$ Categorización: Categorizar las reseñas.

$\cdot$ Análisis de sentimiento: Si al reseña fue positiva, negativa o neutra. SQL no es la mejor opción para realizar este análisis.

##### 5-3 Por que sí usar SQL

Si se cuenta con una gran cantidad de datos y están disponibles directamente en una base relacional. Es una herramienta adecuada para tareas de conteo
o frecuencias, busqueda de palabras clave, limpiar y estructurar datos.

##### 5-4 SQL no es bueno para trabajar con texto

Si la base es muy pequeña, si la consultas requieren ser ser ejecutadas en el menor tiempo posible hay alternativas como Elastic Search, 
si el objetivo es leer todo el texto y entregar un resumen.

##### 5-5 Caracteristicas del Texto


In [2]:
# Mostrar las primeras 5 filas de la tabla disneyrew del esquema analisistexto

query = """
select
    * 
from
    analisistexto.disneyrew
limit
    5;
"""
df = pd.read_sql_query(text(query), engine)

#Con esta sentencia with podemos mostrar todas las columnas y filas del DataFrame sin truncar el contenido de las celdas. Esto es útil cuando queremos inspeccionar los datos completos, especialmente si algunas columnas contienen texto largo o si queremos ver todas las columnas disponibles en el DataFrame.
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head())

,disneyrew_id,Information
0,1,"Rating: 4, Year_Month: 2019-4, Reviewer_Location: Australia, Review_Text: If you've ever been to Disneyland anywhere you'll find Disneyland Hong Kong very similar in the layout when you walk into main street! It has a very familiar feel. One of the rides its a Small World is absolutely fabulous and worth doing. The day we visited was fairly hot and relatively busy but the queues moved fairly well. Branch: Disneyland_HongKong"
1,2,"Rating: 4, Year_Month: 2019-5, Reviewer_Location: Philippines, Review_Text: Its been a while since d last time we visit HK Disneyland .. Yet, this time we only stay in Tomorrowland .. AKA Marvel land!Now they have Iron Man Experience n d Newly open Ant Man n d Wasp!!Ironman .. Great feature n so Exciting, especially d whole scenery of HK (HK central area to Kowloon)!Antman .. Changed by previous Buzz lightyear! More or less d same, but I'm expecting to have something most!!However, my boys like it!!Space Mountain .. Turns into Star Wars!! This 1 is Great!!!For cast members (staffs) .. Felt bit MINUS point from before!!! Just dun feel like its a Disney brand!! Seems more local like Ocean Park or even worst!!They got no SMILING face, but just wanna u to enter n attraction n leave!!Hello this is supposed to be Happiest Place on Earth brand!! But, just really Dont feel it!!Bakery in Main Street now have more attractive delicacies n Disney theme sweets .. These are Good Points!!Last, they also have Starbucks now inside the theme park!! Branch: Disneyland_HongKong"
2,3,"Rating: 4, Year_Month: 2019-4, Reviewer_Location: United Arab Emirates, Review_Text: Thanks God it wasn t too hot or too humid when I was visiting the park otherwise it would be a big issue (there is not a lot of shade).I have arrived around 10:30am and left at 6pm. Unfortunately I didn t last until evening parade, but 8.5 hours was too much for me.There is plenty to do and everyone will find something interesting for themselves to enjoy.It wasn t extremely busy and the longest time I had to queue for certain attractions was 45 minutes (which is really not that bad).Although I had an amazing time, I felt a bit underwhelmed with choice of rides and attractions. The park itself is quite small (I was really expecting something grand even the main castle which was closed by the way was quite small).The food options are good, few coffee shops (including Starbucks) and plenty of gift shops. There was no issue with toilets as they are everywhere.All together it was a great day out and I really enjoyed it. Branch: Disneyland_HongKong"
3,4,"Rating: 4, Year_Month: 2019-4, Reviewer_Location: Australia, Review_Text: HK Disneyland is a great compact park. Unfortunately there is quite a bit of maintenance work going on at present so a number of areas are closed off (including the famous castle) If you go midweek, it is not too crowded and certainly no where near as bus as LA Disneyland. We did notice on this visit that prices for food, drinks etc have really gone through the roof so be prepared to pay top dollar for snacks (and avoid the souvenir shops if you can) Regardless, kids will love it. Branch: Disneyland_HongKong"
4,5,"Rating: 4, Year_Month: 2019-4, Reviewer_Location: United Kingdom, Review_Text: the location is not in the city, took around 1 hour from Kowlon, my kids like disneyland so much, everything is fine. but its really crowded and hot in Hong Kong Branch: Disneyland_HongKong"


In [3]:
# Obtener el numero de caracteres de la columna information */

query = """
select
    length(information) as tamaño
from
    analisistexto.disneyrew
limit
    5;
"""
df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head(10))

,tamaño
0,430
1,1074
2,1051
3,587
4,270


In [4]:
# Agrupar por el numero de caracteres que tienen las reseñas y hacer un conteo

query = """
select
    length(information) as tamaño,
    count(disneyrew_id)
from
    analisistexto.disneyrew
group by
    1
order by
    1 desc;
"""
df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head())

,tamaño,count(disneyrew_id)
0,20860,1
1,18564,1
2,18443,1
3,17466,1
4,17074,1


##### 5-6 Text Parsing 1


In [5]:
# Formula Left y Right para extraer caracteres de una cadena de texto

query = """
select
     left("Reseñas de Disney",7)  as izquierda,
     right("Reseñas de Disney",6) as derecha; 
"""
df = pd.read_sql_query(text(query), engine)
df.head()

,izquierda,derecha
0,Reseñas,Disney


In [6]:
# Extraer los primeros 6 caracteres de la columna de reseñas y contar cuantas veces se repite cada cadena */

query="""
select
    left(information,6) as izquierda,
    count(disneyrew_id)
from
    analisistexto.disneyrew
group by 1;
"""
df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head())

,izquierda,count(disneyrew_id)
0,Rating,42656


In [7]:
# Usar la función right y left para obtener la fecha en que se hizo el comentario. Notemos que faltaría limpiar la fecha.

query="""
select
    right(left(information,30),7) as fecha
from
    analisistexto.disneyrew
group by
    1
order by
    1;
"""
df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head())

,fecha
0,2010-10
1,2010-11
2,2010-12
3,"2010-3,"
4,"2010-4,"


In [8]:
# Función substring_index */
# Esta funcion busca una cadena especificada por el usuario sobre una columa de tipo string y devuelve la cadena antes de ese string o despues de ese string. */
# Si la cadena "un ejemplo" se repite varias veces en el string, la funcion tomara en cuenta solo la primera vez que aparezca */

query="""
select
    substring_index("Esto es un ejemplo de texto","un ejemplo", 1) as Antes, 
    substring_index("Esto es un ejemplo de texto","un ejemplo",-1) as Despues;
"""
df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head())

,Antes,Despues
0,Esto es,de texto


In [9]:
# Usar la funcion substring para extraer la fecha de la columna Infomrmation. Realizar un conteo de cuantas reseñas se hicieron por mes. Notemos que la fecha no esta en formato de fecha, sino como texto.

query = """
select
    substring_index(substring_index(information,"Year_Month: ",-1),",",1) as fecha,
    count(disneyrew_id)
from
    analisistexto.disneyrew
group by
    1
order by
    1;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,fecha,count(disneyrew_id)
0,2010-10,21
1,2010-11,27
2,2010-12,56
3,2010-3,2
4,2010-4,1
5,2010-5,4
6,2010-6,8
7,2010-7,7
8,2010-8,7
9,2010-9,10


##### 5-7 Text Parsing 2

In [ ]:
# Usando la consulta anterior obtener el año y realizar el conteo de numero de registros por año.

query = """
select
    substring_index(substring_index(substring_index(information,"Year_Month: ",-1),",",1),"-",1) as año,
    count(disneyrew_id) as conteo
from
    analisistexto.disneyrew
group by
    1
order by
    1;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)


,año,conteo
0,2010,143
1,2011,1984
2,2012,4342
3,2013,4717
4,2014,5301
5,2015,6979
6,2016,6599
7,2017,5195
8,2018,3997
9,2019,786


In [11]:
# Obtener el año y el mes

query="""
select
     substring_index(substring_index(substring_index(information,"Year_Month: ",-1),",",1),"-",1) as año,
     substring_index(substring_index(substring_index(information,"Year_Month: ",-1),",",1),"-",-1) as mes,
     count(disneyrew_id) as conteo
from
     analisistexto.disneyrew
group by
     1,2
order by
     1;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,año,mes,conteo
0,2010,10,21
1,2010,11,27
2,2010,12,56
3,2010,3,2
4,2010,4,1
5,2010,5,4
6,2010,6,8
7,2010,7,7
8,2010,8,7
9,2010,9,10


In [12]:
# Usar la funcion substring_index para obtener rating, fecha, pais de origen.

query="""
select
     substring_index(substring_index(information,", Year_Month: ",1),"Rating: ",-1) as rating,
     substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1) as fecha,
     substring_index(substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1),"-",1) as año,
     substring_index(substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1),"-",-1) as mes,
     substring_index(substring_index(information,", Review_Text: ",1),", Reviewer_Location: ",-1) as pais_origen,
     substring_index(substring_index(information,", Branch: ",1),", Review_Text: ",-1) as comentario,
     substring_index(Information," Branch: ",-1) as parque
from
     analisistexto.disneyrew;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,rating,fecha,año,mes,pais_origen,comentario,parque
0,4,2019-4,2019,4,Australia,If you've ever been to Disneyland anywhere you...,Disneyland_HongKong
1,4,2019-5,2019,5,Philippines,Its been a while since d last time we visit HK...,Disneyland_HongKong
2,4,2019-4,2019,4,United Arab Emirates,Thanks God it wasn t too hot or too humid wh...,Disneyland_HongKong
3,4,2019-4,2019,4,Australia,HK Disneyland is a great compact park. Unfortu...,Disneyland_HongKong
4,4,2019-4,2019,4,United Kingdom,"the location is not in the city, took around 1...",Disneyland_HongKong
5,3,2019-4,2019,4,Singapore,"Have been to Disney World, Disneyland Anaheim ...",Disneyland_HongKong
6,5,2019-4,2019,4,India,Great place! Your day will go by and you won't...,Disneyland_HongKong
7,3,2019-3,2019,3,Malaysia,Think of it as an intro to Disney magic for th...,Disneyland_HongKong
8,2,2019-4,2019,4,Australia,"Feel so let down with this place,the Disneylan...",Disneyland_HongKong
9,5,2019-3,2019,3,India,I can go on talking about Disneyland. Whatever...,Disneyland_HongKong


##### 5-8 Transformacion de Texto

In [17]:
# Funciones Upper, Lower, Trim

query = """
select
    upper("Hola amigos") as upper,							/* Convierte una cadena a mayusculas */
    lower("Hola amigos") as lower,							/* Convierte una cadena a minusculas */
    trim("    Hola amigos     ") as trim,						/* Elimina espacios en blanco al inicio y final de una cadena */
    trim(leading "$" from "$$Hola amigos") as trim2, 				/* Elimina un caracter inicial que le especifiquemos de una cadena de texto */
    replace("Hola amigos","Hola","Holi") as reemplazar;				/* Permite sustituir una cadena de texto por otra dentro de una string */
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,upper,lower,trim,trim2,reemplazar
0,HOLA AMIGOS,hola amigos,Hola amigos,Hola amigos,Holi amigos


In [24]:
# Convertir la columna rating a entero sin signo.
# Convertir la columna año a entero sin signo.
# Convertir la columna mes a entero sin signo.
# Reemplazar el nombre de país.
# Usar la funcion trim para eliminar espacios en blanco de la columna comentario. 

query = """
SELECT
    CONVERT(rating,unsigned INT) AS rating,
    CASE
        WHEN año="missing" THEN NULL
        ELSE CONVERT(año,UNSIGNED INT)
    END AS año,
    CASE
        WHEN mes="missing" THEN NULL
        ELSE CONVERT(mes,UNSIGNED INT)
    END AS mes,
    REPLACE(REPLACE(REPLACE(pais_origen,"United States","US"),"United Kingdom","UK"),"United Arab Emirates","UAE") AS pais_origen,
    TRIM(comentario) AS comentario,
    parque
FROM
    (
    SELECT
    
        substring_index(substring_index(information,", Year_Month: ",1),"Rating: ",-1) as rating,
        substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1) as fecha,
        substring_index(substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1),"-",1) as año,
        substring_index(substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1),"-",-1) as mes,
        substring_index(substring_index(information,", Review_Text: ",1),", Reviewer_Location: ",-1) as pais_origen,
        substring_index(substring_index(information,", Branch: ",1),", Review_Text: ",-1) as comentario,
        substring_index(information," Branch: ",-1) as parque
    FROM
        analisistexto.disneyrew
) as a;
"""

df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head(10))


,rating,año,mes,pais_origen,comentario,parque
0,4,2019.0,4.0,Australia,If you've ever been to Disneyland anywhere you'll find Disneyland Hong Kong very similar in the layout when you walk into main street! It has a very familiar feel. One of the rides its a Small World is absolutely fabulous and worth doing. The day we visited was fairly hot and relatively busy but the queues moved fairly well. Branch: Disneyland_HongKong,Disneyland_HongKong
1,4,2019.0,5.0,Philippines,"Its been a while since d last time we visit HK Disneyland .. Yet, this time we only stay in Tomorrowland .. AKA Marvel land!Now they have Iron Man Experience n d Newly open Ant Man n d Wasp!!Ironman .. Great feature n so Exciting, especially d whole scenery of HK (HK central area to Kowloon)!Antman .. Changed by previous Buzz lightyear! More or less d same, but I'm expecting to have something most!!However, my boys like it!!Space Mountain .. Turns into Star Wars!! This 1 is Great!!!For cast members (staffs) .. Felt bit MINUS point from before!!! Just dun feel like its a Disney brand!! Seems more local like Ocean Park or even worst!!They got no SMILING face, but just wanna u to enter n attraction n leave!!Hello this is supposed to be Happiest Place on Earth brand!! But, just really Dont feel it!!Bakery in Main Street now have more attractive delicacies n Disney theme sweets .. These are Good Points!!Last, they also have Starbucks now inside the theme park!! Branch: Disneyland_HongKong",Disneyland_HongKong
2,4,2019.0,4.0,UAE,"Thanks God it wasn t too hot or too humid when I was visiting the park otherwise it would be a big issue (there is not a lot of shade).I have arrived around 10:30am and left at 6pm. Unfortunately I didn t last until evening parade, but 8.5 hours was too much for me.There is plenty to do and everyone will find something interesting for themselves to enjoy.It wasn t extremely busy and the longest time I had to queue for certain attractions was 45 minutes (which is really not that bad).Although I had an amazing time, I felt a bit underwhelmed with choice of rides and attractions. The park itself is quite small (I was really expecting something grand even the main castle which was closed by the way was quite small).The food options are good, few coffee shops (including Starbucks) and plenty of gift shops. There was no issue with toilets as they are everywhere.All together it was a great day out and I really enjoyed it. Branch: Disneyland_HongKong",Disneyland_HongKong
3,4,2019.0,4.0,Australia,"HK Disneyland is a great compact park. Unfortunately there is quite a bit of maintenance work going on at present so a number of areas are closed off (including the famous castle) If you go midweek, it is not too crowded and certainly no where near as bus as LA Disneyland. We did notice on this visit that prices for food, drinks etc have really gone through the roof so be prepared to pay top dollar for snacks (and avoid the souvenir shops if you can) Regardless, kids will love it. Branch: Disneyland_HongKong",Disneyland_HongKong
4,4,2019.0,4.0,UK,"the location is not in the city, took around 1 hour from Kowlon, my kids like disneyland so much, everything is fine. but its really crowded and hot in Hong Kong Branch: Disneyland_HongKong",Disneyland_HongKong
5,3,2019.0,4.0,Singapore,"Have been to Disney World, Disneyland Anaheim and Tokyo Disneyland but I feel that Disneyland Hong Kong is really too small to be called a Disneyland. It has way too few rides and attractions. Souvenirs, food and even entrance tickets are slightly more expensive than other Disneyland as well. Basically, this park is good only for small children and people who has never been to Disney. The food choices were acceptable, mostly fast food, and not too expensive. Bottled water, however, was VERY expensive but they do have water fountains around for you to refill your water bottles. The parade was pretty good. It was crowded not a problem but what was the problem was the people were just 

In [35]:
# Con el resultado de la consulta anterior crear una nueva tabla. Esta tabla se llamará disneyclean y estará en el esquema analisistexto.

# Eliminar la tabla si existe
query1 = r"""
DROP TABLE IF EXISTS analisistexto.disneyclean
"""

# Crear la tabla disneyclean con los datos limpios
query2 = """
CREATE TABLE analisistexto.disneyclean AS

SELECT
    CONVERT(rating,UNSIGNED INT) AS rating,
    CASE
        WHEN año="missing" THEN NULL
        ELSE CONVERT(año,UNSIGNED INT)
    END AS año,
    CASE
        WHEN mes="missing" THEN NULL
        ELSE CONVERT(mes,UNSIGNED INT)
    end as mes,
    replace(replace(replace(pais_origen,"United States","US"),"United Kingdom","UK"),"United Arab Emirates","UAE") as pais_origen,
    trim(comentario) as comentario,
    parque
from
    (
    select
        substring_index(substring_index(information,", Year_Month: ",1),"Rating: ",-1) as rating,
        substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1) as fecha,
        substring_index(substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1),"-",1) as año,
        substring_index(substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1),"-",-1) as mes,
        substring_index(substring_index(information,", Review_Text: ",1),", Reviewer_Location: ",-1) as pais_origen,
        substring_index(substring_index(information,", Branch: ",1),", Review_Text: ",-1) as comentario,
        substring_index(information," Branch: ",-1) as parque
    from
        analisistexto.disneyrew
) as a;
"""

with engine.begin() as conn:
    conn.execute(text(query1)) #Ejecutamos cada consulta dentro de la conexión. Esto asegura que ambas consultas se ejecuten en el mismo contexto de transacción.
    conn.execute(text(query2))

In [34]:
# Visualizar las primeras 5 filas de la tabla disneyclean que acabamos de crear

query = r"""
select
    *
from
    analisistexto.disneyclean;
"""
df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head())

,rating,año,mes,pais_origen,comentario,parque
0,4,2019.0,4.0,Australia,If you've ever been to Disneyland anywhere you'll find Disneyland Hong Kong very similar in the layout when you walk into main street! It has a very familiar feel. One of the rides its a Small World is absolutely fabulous and worth doing. The day we visited was fairly hot and relatively busy but the queues moved fairly well. Branch: Disneyland_HongKong,Disneyland_HongKong
1,4,2019.0,5.0,Philippines,"Its been a while since d last time we visit HK Disneyland .. Yet, this time we only stay in Tomorrowland .. AKA Marvel land!Now they have Iron Man Experience n d Newly open Ant Man n d Wasp!!Ironman .. Great feature n so Exciting, especially d whole scenery of HK (HK central area to Kowloon)!Antman .. Changed by previous Buzz lightyear! More or less d same, but I'm expecting to have something most!!However, my boys like it!!Space Mountain .. Turns into Star Wars!! This 1 is Great!!!For cast members (staffs) .. Felt bit MINUS point from before!!! Just dun feel like its a Disney brand!! Seems more local like Ocean Park or even worst!!They got no SMILING face, but just wanna u to enter n attraction n leave!!Hello this is supposed to be Happiest Place on Earth brand!! But, just really Dont feel it!!Bakery in Main Street now have more attractive delicacies n Disney theme sweets .. These are Good Points!!Last, they also have Starbucks now inside the theme park!! Branch: Disneyland_HongKong",Disneyland_HongKong
2,4,2019.0,4.0,UAE,"Thanks God it wasn t too hot or too humid when I was visiting the park otherwise it would be a big issue (there is not a lot of shade).I have arrived around 10:30am and left at 6pm. Unfortunately I didn t last until evening parade, but 8.5 hours was too much for me.There is plenty to do and everyone will find something interesting for themselves to enjoy.It wasn t extremely busy and the longest time I had to queue for certain attractions was 45 minutes (which is really not that bad).Although I had an amazing time, I felt a bit underwhelmed with choice of rides and attractions. The park itself is quite small (I was really expecting something grand even the main castle which was closed by the way was quite small).The food options are good, few coffee shops (including Starbucks) and plenty of gift shops. There was no issue with toilets as they are everywhere.All together it was a great day out and I really enjoyed it. Branch: Disneyland_HongKong",Disneyland_HongKong
3,4,2019.0,4.0,Australia,"HK Disneyland is a great compact park. Unfortunately there is quite a bit of maintenance work going on at present so a number of areas are closed off (including the famous castle) If you go midweek, it is not too crowded and certainly no where near as bus as LA Disneyland. We did notice on this visit that prices for food, drinks etc have really gone through the roof so be prepared to pay top dollar for snacks (and avoid the souvenir shops if you can) Regardless, kids will love it. Branch: Disneyland_HongKong",Disneyland_HongKong
4,4,2019.0,4.0,UK,"the location is not in the city, took around 1 hour from Kowlon, my kids like disneyland so much, everything is fine. but its really crowded and hot in Hong Kong Branch: Disneyland_HongKong",Disneyland_HongKong


##### 5-9 Busquedas con Comodin

In [31]:
# Funcion LIKE

query = r"""
select
    "Daniel Perez Gonzalez" LIKE "Go" as e1,      /* Devuelve 0. Busca coincidencia exacta entre las dos cadenas.*/
    "Daniel Perez Gonzalez" NOT LIKE "Go" as e2,  /* Devuelve 1. Busca la NO coincidencia exacta entre las dos cadenas.*/
    "Daniel Perez Gonzalez" LIKE "%on%" as e3,    /* Devuelve 1. Busca que la cadena empiece con texto cualquiera seguida de la cadena "on" seguida de texto cualquiera.*/
    "Daniel Perez Gonzalez" LIKE "d%" as e4,      /* Devuelve 1. Busca que la cadena empiece con la palabra "d" seguida de texto cualquiera.*/
    "Daniel Perez Gonzalez" LIKE "P%" as e5,      /* Devuelve 0. Busca que la cadena empiece con la palabra "P" seguida de texto cualquiera.*/
    "Daniel Perez Gonzalez" LIKE "%Z" as e6,      /* Devuelve 1. Busca que la cadena empiece con texto cualquiera y termine con "z".*/
    "Daniel Perez Gonzalez" LIKE "%el" as e7,     /* Devuelve 0. Busca que la cadena empiece con texto cualquiera y termine con "el".*/
    "Daniel Perez Gonzalez" LIKE "_a%" as e8,     /* Devuelve 1. Busca que la cadena empiece con un caracter cualquiera, seguida de la letra "a" seguido de texto cualquiera.*/
    "Daniel Perez Gonzalez" LIKE "_e%" as e9;     /* Devuelve 0. Busca que la cadena empiece con un caracter cualquiera, seguida de la letra "e" seguido de texto cualquiera.*/
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,e1,e2,e3,e4,e5,e6,e7,e8,e9
0,0,1,1,1,0,1,0,1,0


In [32]:
# Contar el número de reseñas que contengan la palabra "grandson" */

query = r"""
select
    count(*) 
from
    analisistexto.disneyclean
where
    comentario like "%grandson%";
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,count(*)
0,241


In [33]:
#Contar el número de reseñas que no contengan la palabra "grandson" */

query = r"""
select
    count(*) 
from
    analisistexto.disneyclean
where
    comentario not like "%grandson%";
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,count(*)
0,42415


In [36]:
# Conteo de comentarios qu contegan la palabra son o daughter

query = r"""
select
    count(*)
from
    analisistexto.disneyclean
where
    comentario like "%daughter%" or comentario like "%son%";
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,count(*)
0,9922


In [37]:
# Contar los comentarios que contienen la palabra wife y contienen las palabras son o daugther */

query = r"""
select
    count(*)
from
    analisistexto.disneyclean
where
    comentario like "%wife%" AND (comentario like "%son%" or comentario like "%daughter%" );
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,count(*)
0,540


In [38]:
# Contar los comentarios de acuerdo a si contienen alguna de las siguientes palabras */

query = r"""
select
    case
        when comentario like "%love%" 		then "aman"
        when comentario like "%fabulous%" 	then "fabuloso"
        when comentario like "%like%" 		then "gusta"
        when comentario like "%good%" 		then "bien"
        when comentario like "%bad%" 		then "malo"
        when comentario like "%expensive%" 	then "caro"
        when comentario like "%boring%" 	then "aburrido"
        else "ninguno"
    end as sentimiento,
    count(*)
from
    analisistexto.disneyclean
group by
    1
order by
    1 desc;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,sentimiento,count(*)
0,ninguno,16622
1,malo,947
2,gusta,6907
3,fabuloso,413
4,caro,1853
5,bien,4958
6,aman,10863
7,aburrido,93


In [ ]:
# Obtener la tabla considerando que puede haber comentarios que contenga una o mas palabras.

query="""
select
    comentario like "%love%" as "aman",
    comentario like "%like%" as "gusta",
    comentario like "%bad%"  as "malo",
    count(*)
from analisistexto.disneyclean
group by 1,2,3
order by 1,2,3;
"""
df = pd.read_sql_query(text(query), engine)
df.head(15)

##### 5-10 Busqueda Exacta

In [39]:
# Extraer la primera palabra del comentario y verificar si coincide con una lista de palabras dada */

query = r"""
SELECT
    *
FROM
    (
    SELECT
        substring_index(comentario," ",1) as primera_palabra,
        comentario
    FROM
        analisistexto.disneyclean
    ) as Z
WHERE
    primera_palabra in ("Amazing","Great","Fun","Small","Disgusting");
"""
df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head())

,primera_palabra,comentario
0,Great,Great place! Your day will go by and you won't even know it. Obviously went there for my daughter and she absolutely loved it! Too bad the parade got canceled though. Branch: Disneyland_HongKong
1,Small,"Small disneyland catering to kids, but they have some nice rides including the new Ant Man ride. Pricey food and somewhat long queues but that is part of the theme park. Try to go in the morning and then again in the evening, best is to stay one day at the hotel to maximize the park. Branch: Disneyland_HongKong"
2,Great,Great fun for the family. The fact that it is a smaller park than other Disney s is a plus. Facilities seem new and loads of different games. Branch: Disneyland_HongKong
3,great,"great way to spend a day in Hong kong as a tourist, easy to wander through the park and lots of fun, overall good value, found some good souvenirs along the way..food a little expensive as is with most parks a should do if your over that way Branch: Disneyland_HongKong"
4,Amazing,Amazing The shows Everything was amazing If u are a Disney fan you should visit Is in fact a must and if u didnt do it then ur missing out Branch: Disneyland_HongKong


#### 5-11 Expresiones Regulares

In [40]:
# Busqueda de una palabra dentro de texto

query = r"""
select
    "Los datos son de disney" regexp "Disney" as busqueda,
    "Los datos son de disney" regexp "e" as busqueda;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,busqueda,busqueda
0,1,1


In [41]:
# Negación de busqueda de una palabra dentro de texto

query = r"""
select
    "Los datos son de disney" not regexp "Disney" as busqueda;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,busqueda
0,0


In [42]:
# Busqueda de una palabra que esté contenida en el texto pero no al inicio del texto

query = r"""
select
    "Los datos son de disney" regexp ".disney" as busqueda1,
    "Los datos son de disney" regexp ".los" as busqueda2;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,busqueda1,busqueda2
0,1,0


In [53]:
# Busqueda de palabras con distintas iniciales

query = r"""
select
    "Los datos son de disney" regexp "[abcdef]" as busqueda1,    /*Buscar a o b o c o d o e o f*/
    "Los datos son de disney" regexp "[abcdef]isney" as busqueda2,    /*Buscar aisney o bisney  ... o fisney */
    "Los datos son de disney" regexp "[abcdef]{1}isney" as busqueda3,    /*Buscar la cadena "isney" precedida de 1 caracter entre a y f */
    "Los datos son de disney" regexp "[abcdef]{100,110}isney" as busqueda4    /*Buscar la cadena "isney" precedida de entre 100 y 110 caracteres entre a y f */
;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,busqueda1,busqueda2,busqueda3,busqueda4
0,1,1,1,0


In [54]:
# Buscar cualquier letra entre la a y la z

query = r"""
select
    "Los datos son de disney" regexp "[a-z]" as busqueda;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,busqueda
0,1


In [57]:
# Buscar cualquier numero entre un rango de numeros

query = r"""
select
    "5 minutos en el parque" regexp "[1-5]" as busqueda1,
    "5 minutos en el parque" regexp "[1-3]" as busqueda2,
    "5 minutos en el parque" regexp "[12345]" as busqueda3;   /* Buscar 1 o 2 o 3 o 4 o 5 */
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,busqueda1,busqueda2,busqueda3
0,1,0,1


In [61]:
# Buscar un cadena de 3 digitos cualesquiera entre 0 y 9 en el texto

query = r"""
select
    "El tiempo es de 120 minutos" regexp "[0-9][0-9][0-9]" as busqueda1,    #Solucion buscando 3 veces el patron
    "El tiempo es de 120 minutos" regexp "[0-9]{3}" as busqueda2    #Solución alternativa
    ;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,busqueda1,busqueda2
0,1,1


In [66]:
# Buscar una cadena que empiece con un numero del 0 al 9

query = r"""
select
    "El tiempo es de 120 minutos" regexp "[0-9]+" as busqueda, 		/* El simbolo + indica que buscaremos el patron [0-9] una o más veces en la cadena */
    "El tiempo es de 120 minutos" regexp "^[0-9]+" as busqueda 		/* El simbolo ^ indica que buscaremos el patron [0-9] una o más veces en la cadena y al inicio de la cadena*/
;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,busqueda,busqueda
0,1,0


In [67]:
# Verificar si la frase empieza o termina con una cadena especifica

query = r"""
select
    "Los datos son de Disney" regexp "^los" as busqueda1,     # Si la palabra empieza con "los"
    "Los datos son de Disney" regexp "disney$" as busqueda2;  # Si la palabra termina con "disney"
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,busqueda1,busqueda2
0,1,1


In [69]:
# Busqueda de espacios en blanco en el texto

query = r"""
select
    "Los datos son de Disney" regexp "[:space:]" as busqueda1,
    "Los_datos_son_de_Disney" regexp "[:space:]" as busqueda2;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,busqueda1,busqueda2
0,1,0


In [75]:
# Buscar la palabra "si"

query = r"""
select
    'Silla sillon sinatra' regexp '\\bsi\\b' as busqueda1,
    'Parece que si todo sale bien' regexp '\\bsi\\b' as busqueda2;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,busqueda1,busqueda2
0,0,1


In [81]:
# Busqueda de patrones repetidos n veces */

query = r"""
select
    "La contraseña debe ser de tipo 00a" regexp "[0-9]{2}[a-z]" as busqueda; #Busca el patron de numeros y que se repita 3 veces, y que este seguido por una letra
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,busqueda
0,1


##### 5-12 Expresiones Regulares 2

In [82]:
# Busqueda de comentarios con la cadena "5 estrellas" o "4 estrellas" o ... o "1 estrella"

query = r"""
select
    comentario,
    regexp_substr(comentario,"[0-5] star[s ]") as calificacion    #Laa expresion [s ]	indica que se buscará una letra s o un espacio en blanco
from
    analisistexto.disneyclean
where
    comentario regexp "[0-5] star[s ]";
"""
df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head())

,comentario,calificacion
0,"I love Disney. I can say that I am Donald Duck and Mickey Mouse biggest fan. But I still can't get the idea of spending 2 3 days of your holidays in Disneyland. For that reason, I think the size of the area of HK Disneyland is perfect. It might be slightly smaller than other Disneylands, but for some people who does not want to spend their times in Disneyland (just like me), this place is great. I came to this place in June, when it was 32degrees celsius in HK. If I have to spend my whole day in Disneyland under the crazy sun of HK, I don't think I would make it. Lol. So it might be a good idea not to have a bigger area of Disneyland in 2 seasonal countries such as HK. Furthermore, you still can have all of the favourite rides and attractions. So, overall...i give it 4 stars. Highly recommended if you want to have Disneyland experience but unwilling to spend more than 1 day trip. Branch: Disneyland_HongKong",4 stars
1,All you would expect from a Disney site. The queues were short on the day which meant that we saw rode most attractions in one day. The reason for not giving 5 stars is that an area of the park was to close at 7pm but two of the attractions closed just after 5pm so we could not visit Branch: Disneyland_HongKong,5 stars
2,"This Disneyland is much smaller and therefore, more crowded than the one in California, USA. The best rides in the park are Grizzly Gulch and HyperSpace Mountain. Almost everything else is geared toward very young children. Good visit, but not 5 stars. Branch: Disneyland_HongKong",5 stars
3,A fun space for all age groups... The creativity and the beautiful attractions for kids is totally commendable. The rides are all good . 5 stars for the time managemnt at the shows and rides.... Praiseworthy. We had a lovely time with our kids and they both enjoyed it to the core. Branch: Disneyland_HongKong,5 stars
4,I wish you had something more than 5 star rating ....impossible to express the joy and fun my kids had! Branch: Disneyland_HongKong,5 star


In [83]:
# Con tabla anterior realizar un conteo de acuerdo al numero de estrellas */

query = r"""
select
    regexp_substr(comentario,"[0-5] star[s]") as calificacion,
    count(*)
from
    (
    select
        regexp_replace(comentario,"star[s ,.]","stars") as comentario
    from
        analisistexto.disneyclean
    where
        comentario regexp "[0-5] star[s ,.]"
    ) as A
group by
    1
order by
    2 desc;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,calificacion,count(*)
0,5 stars,169
1,4 stars,58
2,3 stars,31
3,2 stars,21
4,1 stars,10
5,0 stars,4


##### 5-13 Construccion de Texto

In [84]:
# Concatenación de texto

query = r"""
select
    concat(rating," Rating") as rating_2
from
    analisistexto.disneyclean;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,rating_2
0,4 Rating
1,4 Rating
2,4 Rating
3,4 Rating
4,4 Rating
5,3 Rating
6,5 Rating
7,3 Rating
8,2 Rating
9,5 Rating


In [86]:
# Concatenacion de n cadenas de texto indicando un separador */

query = r"""
select
    concat_ws("-",año,mes,pais_origen) as texto
from
    analisistexto.disneyclean;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,texto
0,2019-4-Australia
1,2019-5-Philippines
2,2019-4-UAE
3,2019-4-Australia
4,2019-4-UK
5,2019-4-Singapore
6,2019-4-India
7,2019-3-Malaysia
8,2019-4-Australia
9,2019-3-India


In [87]:
# Para cada parque obtener el pais de origen de los visitantes

query = r"""
select
    parque,
    group_concat(distinct pais_origen) as pais_visitantes
from
    analisistexto.disneyclean
group by
    1;
"""
df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head())

,parque,pais_visitantes
0,Disneyland_California,"Argentina,Armenia,Aruba,Australia,Austria,Bahrain,Bangladesh,Barbados,Belgium,Bolivia,Botswana,Brazil,Bulgaria,Cambodia,Canada,Caribbean Netherlands,Cayman Islands,Chile,China,Colombia,Cook Islands,Costa Rica,Croatia,Cuba,Czechia,Denmark,Dominican Republic,Ecuador,Egypt,Estonia,Ethiopia,Falkland Islands (Islas Malvinas),Fiji,Finland,France,French Polynesia,Germany,Greece,Guam,Guatemala,Honduras,Hong Kong,Hungary,Iceland,India,Indonesia,Iran,Iraq,Ireland,Israel,Italy,Jamaica,Japan,Jersey,Jordan,Kazakhstan,Kenya,Kuwait,Lebanon,Luxembourg,Macau,Malaysia,Malta,Mexico,Montenegro,Morocco,Namibia,Netherlands,New Zealand,Nicaragua,Nigeria,Northern Mariana Islands,Norway,Pakistan,Panama,Papua New Guinea,Peru,Philippines,Poland,Portugal,Puerto Rico,Qatar,Romania,Russia,Rwanda,Saudi Arabia,Singapore,Slovakia,South Africa,South Korea,Spain,Sri Lanka,Sweden,Switzerland,Taiwan,Tanzania,Thailand,The Bahamas,Trinidad and Tobago,Tunisia,Turkey,Turks and Caicos Islands,U.S. Virgin Islands,UAE,Uganda,UK,Ukraine,Uruguay,US,Vanua"
1,Disneyland_HongKong,"Afghanistan,Andorra,Argentina,Australia,Austria,Bahrain,Bangladesh,Barbados,Belgium,Bosnia and Herzegovina,Botswana,Brazil,Brunei,Bulgaria,Cambodia,Canada,Chile,China,Colombia,Cook Islands,Costa Rica,Côte d'Ivoire,Croatia,Curaçao,Cyprus,Czechia,Denmark,Dominican Republic,Egypt,Estonia,Ethiopia,Falkland Islands (Islas Malvinas),Fiji,Finland,France,Germany,Ghana,Gibraltar,Greece,Guam,Guatemala,Haiti,Hong Kong,Hungary,India,Indonesia,Iran,Ireland,Isle of Man,Israel,Italy,Japan,Jersey,Jordan,Kazakhstan,Kenya,Kuwait,Laos,Lithuania,Luxembourg,Macau,Malaysia,Maldives,Mali,Mauritius,Mexico,Mongolia,Montenegro,Mozambique,Myanmar (Burma),Nepal,Netherlands,New Zealand,Nigeria,Northern Mariana Islands,Norway,Oman,Pakistan,Peru,Philippines,Poland,Portugal,Puerto Rico,Qatar,Romania,Russia,Rwanda,Saudi Arabia,Serbia,Seychelles,Singapore,Solomon Islands,South Africa,South Korea,South Sudan,Spain,Sri Lanka,Sweden,Switzerland,Taiwan,Tanzania,Thailand,The Bahamas,Timor-Leste,Trinidad and Tobago,Turkey,UAE,Uganda,UK,Ukraine,Ur"
2,Disneyland_Paris,"Afghanistan,Åland Islands,Albania,Algeria,Andorra,Antigua and Barbuda,Argentina,Australia,Austria,Azerbaijan,Bahrain,Bangladesh,Barbados,Belgium,Bolivia,Bosnia and Herzegovina,Brazil,Brunei,Bulgaria,Cambodia,Canada,Chile,China,Colombia,Croatia,Cyprus,Czechia,Democratic Republic of the Congo,Denmark,Dominican Republic,Ecuador,Egypt,El Salvador,Estonia,Ethiopia,Finland,France,French Polynesia,Georgia,Germany,Gibraltar,Greece,Grenada,Guatemala,Guernsey,Haiti,Honduras,Hong Kong,Hungary,Iceland,India,Indonesia,Iran,Ireland,Isle of Man,Israel,Italy,Jamaica,Japan,Jersey,Jordan,Kazakhstan,Kenya,Kuwait,Laos,Latvia,Lebanon,Libya,Lithuania,Luxembourg,Macau,Madagascar,Malawi,Malaysia,Mali,Malta,Mauritius,Mexico,Moldova,Monaco,Montenegro,Morocco,Mozambique,Myanmar (Burma),Nepal,Netherlands,New Zealand,Nigeria,North Macedonia,Norway,Oman,Pakistan,Panama,Peru,Philippines,Poland,Portugal,Puerto Rico,Qatar,Romania,Russia,Saudi Arabia,Senegal,Serbia,Seychelles,Singapore,Slovakia,Slovenia,Solomon Islands,South Africa,South Kor"
